# Defective-Change Trunk Provenance

The pipeline never checked whether the released **defective change** sits on the reconstructed integration branch. The corrective side is guaranteed to lie on the trunk by construction, whereas the defective side is guaranteed only implicitly. The check covers every released pair, asking whether the defect is first-parent reachable from the trunk tips, meaning it lies on the first-parent mainline, or whether it was reached only through a merge and is therefore off the trunk. Each pair also carries the flag marking defects whose SHA appears in no corpus pull request, together with the audit fate of the off-trunk pairs.

In [1]:
# Released pairs and their corpus-PR flag.
import collections

import polars as pl
import pygit2
from google.cloud import storage

from benchmark.attribution.history import VersionControl
from benchmark.definitions.partitions import Shard
from benchmark_notebook.io import CloudStorage
from benchmark_notebook import theme  # noqa: F401  registers thesis template

store = CloudStorage(bucket="bug-request-bucket")

# The localization-inducer-reference asset is the full released set, one row per
# released defect, and it already carries the direct flag: true when no corpus pull
# request contains the defect SHA. The has_pr column is its negation, so the second
# property needs no extra work and no GitHub access.
pairs = (
    pl.from_arrow(
        store.asset(
            key="localization-inducer-reference", version="3.1.0", partition="default"
        )
    )
    .select("name", "correction", "defect", "direct")
    .with_columns((~pl.col("direct")).alias("has_pr"))
)

print(f"released pairs: {pairs.height}")
print(f"repositories:   {pairs['name'].n_unique()}")
print(f"has_pr=True:     {int(pairs['has_pr'].sum())}")
print(f"has_pr=False:    {int((~pairs['has_pr']).sum())}")
pairs.head()

released pairs: 745
repositories:   6
has_pr=True:     712
has_pr=False:    33


name,correction,defect,direct,has_pr
str,str,str,bool,bool
"""microsoft/Olive""","""00038331d0294e8b2304d28cf66500…","""3d5bc7183ee938cd3a6bbb88534245…",true,false
"""kedro-org/kedro-plugins""","""004888c9be93ba3eb2e450828418ff…","""3dac0d4da476f4b014e5bababfcf96…",false,true
"""microsoft/Olive""","""01a5cb9bebb22f87aef572369d820c…","""b311be67e1e9d0b95fb547a34c728e…",false,true
"""microsoft/Olive""","""01a9755692acb0a481b15075ddd869…","""ff14ee7ab870f3127a76ca672222d3…",false,true
"""microsoft/Olive""","""01dbc53c8f0d1cd057f320db105303…","""1e7da9de8666793b1478ffd4d983bd…",false,true


In [2]:
# Trunk branch names per repository, and the full cohort name space.
stationary = pl.from_arrow(
    store.asset(
        key="scope-repository-stationary", version="1.0.0", partition="default"
    )
)

# The scope-source-raw partition token is computed over the whole stationary
# cohort, so the name list must be the full sorted set of repositories even though
# only a handful of them carry released pairs. Using the pair subset would compute
# the wrong partition token.
names = sorted(stationary["name"].to_list())
trunks = {
    row["name"]: list(row["trunks"] or [])
    for row in stationary.select("name", "trunks").iter_rows(named=True)
}

repositories = sorted(pairs["name"].unique().to_list())
print(f"cohort repositories:        {len(names)}")
print(f"repositories with pairs:    {len(repositories)}")
for name in repositories:
    print(f"  {name}: trunks={trunks.get(name)}")

cohort repositories:        130
repositories with pairs:    6
  OpenJobDescription/openjd-model-for-python: trunks=['mainline']
  awslabs/graphstorm: trunks=['main']
  cohere-ai/cohere-compass-sdk: trunks=['main']
  kedro-org/kedro-plugins: trunks=['main']
  microsoft/Olive: trunks=['main', 'master']
  mistralai/mistral-common: trunks=['main']


In [3]:
# Reconstruct the trunk commit set per repository.
#
# The trunk is the first-parent history of its integration-branch tips. For each
# trunk branch, the tip is resolved and walked topologically along first parents,
# collecting every commit on that chain. This mirrors the commit-attribution
# procedure, so a defect is on the trunk exactly when its SHA lands in this set.
# Reconstructing the set here, rather than reading the per-repository,
# GitHub-backed attribution asset, keeps the check self-contained.
client = storage.Client(project="bug-request")


def trunk_commits(name: str) -> set[str]:
    control = VersionControl.create(
        client,
        bucket="bug-request-bucket",
        key="scope-source-raw",
        location="main",
        version="4.0.0",
        partition=Shard.to_space(names, name),
    )
    git = control.handle
    seen: set[str] = set()
    for branch in trunks.get(name, []):
        tip = None
        for ref in (f"refs/heads/{branch}", f"refs/remotes/origin/{branch}"):
            try:
                tip = str(git.lookup_reference(ref).peel(pygit2.Commit).id)
                break
            except KeyError:
                continue
        if tip is None:
            print(f"  {name}: branch {branch} has no reference, skipping")
            continue
        walker = git.walk(tip, pygit2.GIT_SORT_TOPOLOGICAL)
        walker.simplify_first_parent()
        for commit in walker:
            seen.add(str(commit.id))
    return seen


trunk_sets = {}
for name in repositories:
    trunk_sets[name] = trunk_commits(name)
    print(f"{name}: {len(trunk_sets[name])} trunk commits")

OpenJobDescription/openjd-model-for-python: 204 trunk commits
awslabs/graphstorm: 811 trunk commits
cohere-ai/cohere-compass-sdk: 167 trunk commits
kedro-org/kedro-plugins: 616 trunk commits
  microsoft/Olive: branch master has no reference, skipping
microsoft/Olive: 1688 trunk commits
mistralai/mistral-common: 145 trunk commits


In [4]:
# Property join: is each defect on the trunk?
checked = pairs.with_columns(
    pl.struct("name", "defect")
    .map_elements(
        lambda row: row["defect"] in trunk_sets.get(row["name"], set()),
        return_dtype=pl.Boolean,
    )
    .alias("on_trunk")
)

# Cross-tabulation of on_trunk against has_pr over every released pair.
crosstab = (
    checked.group_by("on_trunk", "has_pr")
    .len()
    .with_columns((pl.col("len") / checked.height).round(4).alias("share"))
    .sort("on_trunk", "has_pr", descending=[True, True])
)

assert crosstab["len"].sum() == checked.height, "cross-tab lost rows"
print(f"pairs: {checked.height}")
print(f"on_trunk=True:  {int(checked['on_trunk'].sum())}")
print(f"on_trunk=False: {int((~checked['on_trunk']).sum())}")
crosstab

pairs: 745
on_trunk=True:  674
on_trunk=False: 71


on_trunk,has_pr,len,share
bool,bool,u32,f64
true,true,641,0.8604
true,false,33,0.0443
false,true,71,0.0953


In [5]:
# Flagged pairs. A defect that is off-trunk or has no corpus pull request violates
# the provenance that the corrective side is guaranteed by construction.
flagged = checked.filter(~pl.col("on_trunk") | ~pl.col("has_pr")).sort(
    "name", "correction", "defect"
)

by_repo = (
    checked.group_by("name")
    .agg(
        pl.len().alias("pairs"),
        (~pl.col("on_trunk")).sum().alias("off_trunk"),
        (~pl.col("has_pr")).sum().alias("no_pr"),
    )
    .sort("name")
)

print(f"flagged pairs (off-trunk or no-PR): {flagged.height} / {checked.height}")
print(by_repo)
flagged

flagged pairs (off-trunk or no-PR): 104 / 745
shape: (6, 4)
┌─────────────────────────────────┬───────┬───────────┬───────┐
│ name                            ┆ pairs ┆ off_trunk ┆ no_pr │
│ ---                             ┆ ---   ┆ ---       ┆ ---   │
│ str                             ┆ u32   ┆ u32       ┆ u32   │
╞═════════════════════════════════╪═══════╪═══════════╪═══════╡
│ OpenJobDescription/openjd-mode… ┆ 23    ┆ 0         ┆ 2     │
│ awslabs/graphstorm              ┆ 236   ┆ 51        ┆ 0     │
│ cohere-ai/cohere-compass-sdk    ┆ 32    ┆ 0         ┆ 12    │
│ kedro-org/kedro-plugins         ┆ 80    ┆ 1         ┆ 11    │
│ microsoft/Olive                 ┆ 349   ┆ 7         ┆ 4     │
│ mistralai/mistral-common        ┆ 25    ┆ 12        ┆ 4     │
└─────────────────────────────────┴───────┴───────────┴───────┘


name,correction,defect,direct,has_pr,on_trunk
str,str,str,bool,bool,bool
"""OpenJobDescription/openjd-mode…","""4c2c2535f3202a64adce3202845336…","""cd0e2892174b0fbff46e0c7a220f6a…",true,false,true
"""OpenJobDescription/openjd-mode…","""eed64c9fa837b7b5df5ee366862cfd…","""bc8f842a1964f25a07e7cb399a6d87…",true,false,true
"""awslabs/graphstorm""","""063fd7576e73a8113df13c1ab8ab00…","""2777f2a566bbb615648b59e28bdfff…",false,true,false
"""awslabs/graphstorm""","""0e1ad7510f4523795bbf8234479b78…","""2777f2a566bbb615648b59e28bdfff…",false,true,false
"""awslabs/graphstorm""","""18074f6fd69d13408ca4c8577276e1…","""2777f2a566bbb615648b59e28bdfff…",false,true,false
…,…,…,…,…,…
"""mistralai/mistral-common""","""a52eab2dfe390977730a6291e91e1b…","""5f95e3ae461ad95fad8f4b7c5e7360…",false,true,false
"""mistralai/mistral-common""","""ab2fcafce664bb973641a1870de060…","""c726553ca4bdfa4c78e47bb31fe452…",false,true,false
"""mistralai/mistral-common""","""b9e3bc05419986b9d57713f32780cb…","""dcfbcb7cdb6a2e44e2b68651b6391e…",false,true,false


## Audit fate

How many pairs reached the realised audit, split by trunk and pull-request provenance, and how the audited ones were rated. The stage outcome is recomputed here from the raw annotation fields rather than read from the stored verdict.

In [6]:
# Audit sample membership and per-pair rating.
sample = pl.from_arrow(
    store.asset(key="audit-change-sample", version="4.0.0", partition="default")
).select("name", "correction", "defect", "rho", "rho_bin")

annotation = pl.from_arrow(
    store.asset(
        key="audit-change-annotation", version="1.0.0", location="main", partition="default"
    )
).rename({"repository": "name", "introducer": "defect"})

# Recompute the stage outcome from the raw fields. The stored verdict is ignored.
annotation = annotation.with_columns(
    pl.when(~pl.col("categories").list.contains("intrinsic"))
    .then(pl.lit("stage_1"))
    .when(pl.col("attribution") == "wrong")
    .then(pl.lit("stage_2"))
    .otherwise(pl.lit("passed"))
    .alias("stage")
)

# The annotation asset carries the correction pull-request number, not the
# correction SHA, so ratings are aggregated over name and defect, the annotation
# grain, and the sample supplies the correction. A minimum of three submissions can
# put several annotator rows on a pair, so the annotator stages are collected into a
# list, then resolved to the majority stage, with ties broken towards the more
# severe outcome.
severity = {"stage_1": 0, "stage_2": 1, "passed": 2}


def majority_stage(stages: list[str]) -> str:
    counts = collections.Counter(stages)
    return min(counts.items(), key=lambda kv: (-kv[1], severity[kv[0]]))[0]


rating = (
    annotation.group_by("name", "defect")
    .agg(
        pl.col("stage").alias("stages"),
        pl.col("attribution").mode().first().alias("attribution"),
        pl.col("confidence").mean().alias("confidence"),
        pl.len().alias("annotators"),
    )
    .with_columns(
        pl.col("stages")
        .map_elements(majority_stage, return_dtype=pl.Utf8)
        .alias("stage")
    )
    .drop("stages")
)

# The sample rows key on the full name, correction and defect. Every one matches a
# released pair on that key, so the audited frame is exactly the 100 audited pairs.
audited = (
    sample.join(rating, on=["name", "defect"], how="left")
    .with_columns(pl.lit(True).alias("in_audit"))
)

print(f"realised sample: {sample.height}")
print(f"rated pairs:     {rating.height}")
print(f"annotators per pair: {sorted(rating['annotators'].unique().to_list())}")
audited.head()

realised sample: 100
rated pairs:     100
annotators per pair: [1]


name,correction,defect,rho,rho_bin,attribution,confidence,annotators,stage,in_audit
str,str,str,f64,i64,str,f64,u32,str,bool
"""microsoft/Olive""","""847d3feb1b975ccc0b2c2f025818af…","""cd640aeebeb0bb128568f37eb49b31…",0.312059,0,"""correct""",3.0,1,"""passed""",true
"""microsoft/Olive""","""a7a70dc758f6acc2e8467039ccf079…","""ff929df6a3bd5b3082641f5ce34b72…",0.319111,0,"""correct""",3.0,1,"""passed""",true
"""awslabs/graphstorm""","""10d229c061778ec660562fc165bf72…","""5e704dc7e7e3a28f12a1ae419abfd9…",0.318406,0,null,3.0,1,"""stage_1""",true
"""microsoft/Olive""","""b86f03c57bb6b92dded618daaac963…","""38bd1d604ec686e6bc4264d234d222…",0.21756,0,"""correct""",3.0,1,"""passed""",true
"""microsoft/Olive""","""06cd412ce3699ce503b0d2d95b51f5…","""b6289263e9321b185f973029ede7e9…",0.149154,0,null,3.0,1,"""stage_1""",true


In [7]:
# Join audit fate onto every released pair, then split by property.
#
# Membership is a fact about the full name, correction and defect. A defect can be
# released under several corrections, but only the sampled pair was audited, so the
# join uses the full key. Pairs that were not sampled get in_audit set to false.
membership = audited.select(
    "name", "correction", "defect", "in_audit", "rho", "rho_bin",
    "stage", "attribution", "confidence"
)
population = (
    checked.join(membership, on=["name", "correction", "defect"], how="left")
    .with_columns(pl.col("in_audit").fill_null(False))
)

# How many of each cohort reached the audit.
def audit_rate(frame: pl.DataFrame, label: str) -> dict:
    n = frame.height
    entered = int(frame["in_audit"].sum())
    return {"cohort": label, "pairs": n, "in_audit": entered,
            "share": round(entered / n, 4) if n else 0.0}


coverage = pl.DataFrame([
    audit_rate(population, "all released"),
    audit_rate(population.filter(~pl.col("on_trunk")), "off-trunk"),
    audit_rate(population.filter(~pl.col("has_pr")), "no-PR"),
    audit_rate(population.filter(pl.col("on_trunk") & pl.col("has_pr")), "on-trunk & has-PR"),
])

assert int(population["in_audit"].sum()) == sample.height, "audit membership mismatch"
print(coverage)
coverage

shape: (4, 4)
┌───────────────────┬───────┬──────────┬────────┐
│ cohort            ┆ pairs ┆ in_audit ┆ share  │
│ ---               ┆ ---   ┆ ---      ┆ ---    │
│ str               ┆ i64   ┆ i64      ┆ f64    │
╞═══════════════════╪═══════╪══════════╪════════╡
│ all released      ┆ 745   ┆ 100      ┆ 0.1342 │
│ off-trunk         ┆ 71    ┆ 2        ┆ 0.0282 │
│ no-PR             ┆ 33    ┆ 0        ┆ 0.0    │
│ on-trunk & has-PR ┆ 641   ┆ 98       ┆ 0.1529 │
└───────────────────┴───────┴──────────┴────────┘


cohort,pairs,in_audit,share
str,i64,i64,f64
"""all released""",745,100,0.1342
"""off-trunk""",71,2,0.0282
"""no-PR""",33,0,0.0
"""on-trunk & has-PR""",641,98,0.1529


In [8]:
# How were the audited pairs rated, split by provenance.
audited_pairs = population.filter(pl.col("in_audit")).with_columns(
    pl.when(pl.col("on_trunk") & pl.col("has_pr"))
    .then(pl.lit("clean"))
    .otherwise(pl.lit("flagged"))
    .alias("provenance")
)

rating_breakdown = (
    audited_pairs.group_by("provenance", "stage")
    .agg(pl.len().alias("pairs"), pl.col("confidence").mean().round(2).alias("mean_confidence"))
    .sort("provenance", "stage")
)

# Any flagged pair that was audited, meaning off-trunk or without a corpus pull
# request, together with its outcome.
flagged_audited = audited_pairs.filter(pl.col("provenance") == "flagged").select(
    "name", "correction", "defect", "on_trunk", "has_pr", "stage", "attribution",
    "confidence", "rho", "rho_bin"
).sort("name", "defect")

print(rating_breakdown)
print(f"\nflagged pairs that were audited: {flagged_audited.height}")
flagged_audited

shape: (5, 4)
┌────────────┬─────────┬───────┬─────────────────┐
│ provenance ┆ stage   ┆ pairs ┆ mean_confidence │
│ ---        ┆ ---     ┆ ---   ┆ ---             │
│ str        ┆ str     ┆ u32   ┆ f64             │
╞════════════╪═════════╪═══════╪═════════════════╡
│ clean      ┆ passed  ┆ 82    ┆ 2.72            │
│ clean      ┆ stage_1 ┆ 11    ┆ 2.8             │
│ clean      ┆ stage_2 ┆ 5     ┆ 3.0             │
│ flagged    ┆ passed  ┆ 1     ┆ 3.0             │
│ flagged    ┆ stage_1 ┆ 1     ┆ 3.0             │
└────────────┴─────────┴───────┴─────────────────┘

flagged pairs that were audited: 2


name,correction,defect,on_trunk,has_pr,stage,attribution,confidence,rho,rho_bin
str,str,str,bool,bool,str,str,f64,f64,i64
"""kedro-org/kedro-plugins""","""71f67b60902bae304f79cad20ca197…","""904fea2becb44df6804597b18a32f9…",false,true,"""stage_1""",null,3.0,0.504584,1
"""mistralai/mistral-common""","""535b4d0a0fc94674ea17db6cf8dc20…","""dcfbcb7cdb6a2e44e2b68651b6391e…",false,true,"""passed""","""correct""",3.0,0.70134,2


In [14]:
# Provenance quadrants of the released pairs.
import plotly.graph_objects as go

quad = {
    (True, True): "on-trunk & has-PR",
    (True, False): "on-trunk, no PR",
    (False, True): "off-trunk, has PR",
    (False, False): "off-trunk, no PR",
}
counts = {
    label: checked.filter(
        (pl.col("on_trunk") == ot) & (pl.col("has_pr") == pr)
    ).height
    for (ot, pr), label in quad.items()
}

go.Figure(
    go.Bar(
        x=list(counts.keys()),
        y=list(counts.values()),
        text=list(counts.values()),
        textposition="outside",
    )
).update_layout(
    xaxis_title="provenance quadrant",
    yaxis_title="pairs",
    template="thesis",
    showlegend=False,
).show()

In [12]:
# Audit coverage per cohort. Share of each provenance group that reached the audit.
cohorts = coverage["cohort"].to_list()
shares = coverage["share"].to_list()
labels = [f"{e}/{n}" for e, n in zip(coverage["in_audit"], coverage["pairs"])]

go.Figure(
    go.Bar(x=cohorts, y=shares, text=labels, textposition="outside")
).update_layout(
    xaxis_title="cohort",
    yaxis_title="share entering the audit",
    yaxis_tickformat=".0%",
    template="thesis",
    showlegend=False,
).show()